In [1]:
import pandas as pd
import keras
from pathlib import Path
from EvolutionEnvironment import EvolutionEnvironment
from Organism import Organism

In [2]:
path = Path("../../basic_ml/diabetes.csv")
path = path.resolve()
print(path)

/home/baile/ml_evolution/basic_ml/diabetes.csv


In [3]:
df = pd.read_csv(path)
y = df[df.columns[-1]].values
x = df.drop(columns=[df.columns[-1]]).values

In [4]:
test_model = keras.Sequential(layers=[
    keras.layers.Input(shape=(8,)),
    keras.layers.Dense(units=32, activation='relu'),
    keras.layers.Dense(units=16, activation='relu'),
    keras.layers.Dense(units=16, activation='relu'),
    keras.layers.Dense(units=16, activation='relu'),
    keras.layers.Dense(units=16, activation='relu'),
    keras.layers.Dense(units=16, activation='relu'),
    keras.layers.Dense(units=16, activation='relu'),
    keras.layers.Dense(units=8, activation='leaky_relu'),
    keras.layers.Dense(units=1, activation='leaky_relu'),
])


In [5]:
layer = keras.layers.Dense(units=16, activation='relu')
layer.build(input_shape=(None, 8))
print(layer.get_weights()[0].shape)

(8, 16)


In [6]:
test = EvolutionEnvironment(test_model, x, y, epochs_per_cycle=10, learning_rate=0.005) # type: ignore
test.init_sim()

I0000 00:00:1793585809.101673    3241 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793585809.204193    3241 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793585810.770219    3241 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [6]:
import numpy as np
from random import random
weights:list[np.ndarray] = test.orgs[0].model.layers[2].get_weights()
weights[1] = np.append(weights[1], 0)
rng = np.random.default_rng()
weights[0] = np.append(weights[0], rng.random((weights[0].shape[1],)))
weights[0] = weights[0].reshape(17, 16)
#weights[0] = np.append(weights[0], (np.array([random() for _ in range(weights[0].shape[1])])))

In [ ]:
print(test.orgs[0].model.layers[2].get_weights()[0].shape,weights[0].shape)
print(test.orgs[0].model.layers[2].get_weights()[0] == weights[0][:16])
print(weights[1].shape)

In [8]:
parent_layers = [layer.shape for layer in test.orgs[0].model.get_weights()]
print(parent_layers)
new_organism = Organism(test.orgs[0].model, test, False)
new_org_layers = [layer.shape for layer in new_organism.model.get_weights()]
print(f"Equivalent: {parent_layers == new_org_layers}")
if parent_layers != new_org_layers:
    print(new_org_layers)

[(8, 32), (32,), (32, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 8), (8,), (8, 1), (1,)]
4
adding_neuron
16 <function relu at 0x76f47c5f1b20>
(16, 16) (16, 17)
Equivalent: False
[(8, 32), (32,), (32, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 17), (17,), (16, 16), (16,), (16, 16), (16,), (16, 8), (8,), (8, 1), (1,)]


In [11]:
print(f"Before {1} Cycle:\nOrganisms: {test.orgs}")
for i, org in enumerate(test.orgs):
    if org != None:
        print(f"Organism {i+1} # Layers: {len(org.model.layers)} Shape of Layers: {[layer.shape for layer in org.model.get_weights()]}")
        print(f"Accury on train data: {org.model.evaluate(test.x_train, test.y_train)[1]:.5f}\nAccuracy on test data: {org.model.evaluate(test.x_test, test.y_test)[1]:.5f}")

test.cycle()
print(f"After {1} Cycle:\nOrganisms: {test.orgs}")
for i, org in enumerate(test.orgs):
    if org != None:
        print(f"Organism {i+1} # Layers: {len(org.model.layers)} Shape of Layers: {[layer.shape for layer in org.model.get_weights()]}")
        print(f"Accury on train data: {org.model.evaluate(test.x_train, test.y_train)[1]:.5f}\nAccuracy on test data: {org.model.evaluate(test.x_test, test.y_test)[1]:.5f}")

Before 1 Cycle:
Organisms: [<Organism.Organism object at 0x71d93e65f560>, <Organism.Organism object at 0x71d8fe3d83e0>, <Organism.Organism object at 0x71d8fde15460>, <Organism.Organism object at 0x71d8fd2a89b0>, <Organism.Organism object at 0x71d97cad1d90>, <Organism.Organism object at 0x71d8fd26ec60>, <Organism.Organism object at 0x71d8fbe3b980>, <Organism.Organism object at 0x71d93dd31490>]
Organism 1 # Layers: 9 Shape of Layers: [(8, 32), (32,), (32, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 8), (8,), (8, 1), (1,)]
 1/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - Accuracy: 0.8750 - loss: 0.0806

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - Accuracy: 0.7557 - loss: 0.1728
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - Accuracy: 0.7208 - loss: 0.2314
Accury on train data: 0.75570
Accuracy on test data: 0.72078
Organism 2 # Layers: 9 Shape of Layers: [(8, 32), (32,), (32, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 8), (8,), (8, 1), (1,)]
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - Accuracy: 0.7622 - loss: 0.1703
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - Accuracy: 0.7013 - loss: 0.2315
Accury on train data: 0.76221
Accuracy on test data: 0.70130
Organism 3 # Layers: 9 Shape of Layers: [(8, 32), (32,), (32, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 16), (16,), (16, 8), (8,), (8, 1), (1,)]
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - Accuracy: 0.7769 - loss: 0.1638
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - Accuracy: 0.7468 - loss: 0.2265
Accury on train data: 0.77687
Accuracy on test data: 0.74675
Organism 

In [ ]:
### Learning rate testing
def write_losses(num_epochs:list[int], all_losses:list[dict]):
    with open("losses.txt", "w", encoding="utf-8") as f: 
        for i, epochs in enumerate(num_epochs):
            f.write(f"# of Epochs: {epochs}\n") 
            losses = all_losses[i]
            for rate in losses.keys():
                f.write(f"\tRate: {rate}, Loss: {losses[rate][0]}, Accuracy: {losses[rate][1]}\n")
            f.write("\n\n")

num_epochs = [10, 20, 30, 40]
all_losses = []
for epochs in num_epochs:
    test.epochs_per_cycle = epochs

    losses = dict()
    learning_rates =  [1*10**-5, 5*10**-5, 1*10**-4, 5*10**-4, 1*10**-3, 5*10**-3, 0.01, 0.05, 0.1, 0.5, 1., 2., 4., 7., 10.]
    for learning_rate in learning_rates:
        test.learning_rate = learning_rate
        test.init_sim()
        test.cycle()
        losses[learning_rate] = test.orgs[0].model.evaluate(test.x_test, test.y_test, batch_size=24)
    all_losses.append(losses)

write_losses(num_epochs, all_losses)